In [ ]:
import time
from alphagenome import colab_utils
from alphagenome.data import genome
from alphagenome.models import dna_client, variant_scorers
import pandas as pd
from tqdm import tqdm
import os
from datetime import datetime, timedelta
import signal
import sys
import numpy as np

# ============================================================================
# CONFIGURATION - Edit these variables
# ============================================================================

TOTAL_WORKERS = 35  # Total number of workers
VCF_PATH = '1000G_SNPs_hg38.tsv'  # Path to your VCF file
BATCH_SIZE = 25  # Number of variants per batch
RETRY_DELAY = 1.23  # Seconds to wait before retrying failed batches
BUFFER_TIME = 2  # Minutes to reserve before time limit for graceful shutdown


# ============================================================================
# FUNCTIONS
# ============================================================================

def load_processed_variants(output_path):
    """Load processed variants from both .csv and .csv.corrupted.bak, combine, and create single backup."""
    processed_variants = set()
    header_written = False

    # List possible files to load
    files_to_load = [output_path, output_path + '.corrupted.bak']
    
    for file_path in files_to_load:
        if not os.path.exists(file_path):
            continue
        try:
            existing_df = pd.read_csv(file_path)
            if 'variant_name' in existing_df.columns:
                processed_variants.update(existing_df['variant_name'].unique())
                header_written = True
                print(f"Loaded {len(existing_df)} variants from {file_path}", flush=True)
        except Exception as e:
            print(f"Warning: Could not load {file_path} normally: {e}", flush=True)
            try:
                print(f"Attempting to recover partial data from {file_path}...", flush=True)
                with open(file_path, 'r') as f:
                    lines = f.readlines()
                    if len(lines) > 0:
                        header_line = lines[0].strip()
                        header_parts = header_line.split(',')
                        try:
                            variant_name_idx = header_parts.index('variant_name')
                            header_written = True
                            for line in lines[1:]:
                                parts = line.strip().split(',')
                                if len(parts) > variant_name_idx:
                                    processed_variants.add(parts[variant_name_idx])
                        except ValueError:
                            print("Could not find variant_name column, skipping file", flush=True)
            except Exception as e2:
                print(f"Could not recover data from {file_path}: {e2}", flush=True)
    
    print(f"Total processed variants loaded: {len(processed_variants)}", flush=True)
    
    # Optionally, write out a combined backup file
    if processed_variants:
        backup_path = output_path + '.consolidated.bak'
        with open(backup_path, 'w') as out:
            out.write('variant_name\n')
            for variant in processed_variants:
                out.write(f'{variant}\n')
        print(f"Wrote consolidated backup to {backup_path}", flush=True)
        header_written = True  # Header present in new backup

    return processed_variants, header_written


# ============================================================================
# MAIN PROCESSING
# ============================================================================

# Initialize (assuming you have these imports)
# import dna_client
# import variant_scorers
# import genome

# Load VCF
vcf = pd.read_csv(VCF_PATH, sep='\t')
vcf['CHROM'] = 'chr' + vcf['chrom'].astype(str)
switch_colnames = dict(zip(['pos', 'ref', 'alt', 'rsid'], ['POS', 'REF', 'ALT', 'variant_id']))
vcf.rename(columns=switch_colnames, inplace=True)
vcf['variant_name'] = vcf['CHROM'].astype(str) + ':' + vcf['POS'].astype(str) + ':' + vcf['REF'] + '>' + vcf['ALT']

# Split by worker
total_rows = len(vcf)
rows_per_worker = total_rows // TOTAL_WORKERS
start_idx = WORKER * rows_per_worker
end_idx = total_rows if WORKER == TOTAL_WORKERS - 1 else start_idx + rows_per_worker
vcf_subset = vcf.iloc[start_idx:end_idx].reset_index(drop=True)
vcf_subset = vcf_subset[vcf_subset['POS']>=0]

print(f"Processing rows {start_idx} to {end_idx} ({len(vcf_subset)} variants)\n", flush=True)

# Setup
organism = dna_client.Organism.HOMO_SAPIENS
sequence_length = dna_client.SUPPORTED_SEQUENCE_LENGTHS['SEQUENCE_LENGTH_1MB']
all_scorers = variant_scorers.RECOMMENDED_VARIANT_SCORERS
selected_scorers = list(all_scorers.values())
output_path = OUTPUT_PATH or f'./results/worker_{WORKER}.csv'

# Create output directory if it doesn't exist
os.makedirs(os.path.dirname(output_path) if os.path.dirname(output_path) else '.', exist_ok=True)

# Load already processed variants
processed_variants, header_written = load_processed_variants(output_path)

if processed_variants:
    vcf_subset = vcf_subset[~vcf_subset['variant_name'].isin(processed_variants)].reset_index(drop=True)
    print(f"Remaining: {len(vcf_subset) - len(processed_variants)} variants to process\n", flush=True)

variants_processed = len(processed_variants)

Time limit: 590 minutes
Will stop gracefully at: 2025-11-29 04:30:12
Buffer time: 2 minutes

WORKER 8/35 - Using API key #0



Processing rows 1958640 to 2203470 (244691 variants)

Loaded 244691 variants from ./results/worker_8.csv
Total processed variants loaded: 244691
Wrote consolidated backup to ./results/worker_8.csv.consolidated.bak
Remaining: -244691 variants to process



In [9]:
import pandas as pd

dfs = []
for i in range(35):
    df = pd.read_csv(f'./results/worker_{i}.csv')
    dfs.append(df)
df = pd.concat(dfs)
df

,variant_name,ATAC_quantile,CAGE_quantile,CHIP_HISTONE_quantile,CHIP_TF_quantile,CONTACT_MAPS_quantile,DNASE_quantile,PROCAP_quantile,RNA_SEQ_quantile,SPLICE_JUNCTIONS_quantile,...,CAGE_raw,CHIP_HISTONE_raw,CHIP_TF_raw,CONTACT_MAPS_raw,DNASE_raw,PROCAP_raw,RNA_SEQ_raw,SPLICE_JUNCTIONS_raw,SPLICE_SITES_raw,SPLICE_SITE_USAGE_raw
0,chr1:11008:C>G,0.952545,0.933411,0.984525,0.995798,0.954588,0.986803,0.933411,0.996574,NaN,...,8.304840,8293.750,2734.125,0.001808,257.218500,6.853630,2.917312,NaN,NaN,NaN
1,chr1:11012:C>G,0.958450,0.963724,0.999058,0.999681,0.974553,0.988931,0.913094,0.997151,NaN,...,8.553116,8275.500,2761.000,0.002071,263.801150,6.618187,2.849740,NaN,NaN,NaN
2,chr1:13110:G>A,0.628119,0.786165,0.964537,0.971090,0.810542,0.822706,0.883945,0.999703,NaN,...,0.888182,6460.000,990.500,0.001369,33.057830,7.526024,2.690983,NaN,NaN,NaN
3,chr1:13116:T>G,0.568981,0.768479,0.982497,0.940451,0.958425,0.692978,0.883945,0.999198,NaN,...,0.893719,6506.000,1038.000,0.001640,32.842957,7.535484,2.784506,NaN,NaN,NaN
4,chr1:13118:A>G,0.876682,0.784491,0.947050,0.925572,0.938341,0.856617,0.900830,0.999580,NaN,...,0.946449,6600.000,1094.000,0.001701,34.294495,8.073830,2.789948,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
244227,chr22:50798635:T>C,0.868425,0.995975,0.991597,0.990788,0.965955,0.934881,0.963724,0.999887,NaN,...,3.709938,867.250,282.750,0.001926,7.350131,2.703756,12.112911,NaN,NaN,NaN
244228,chr22:50798936:A>G,0.884452,0.980378,0.991788,0.995687,0.940957,0.984389,0.879325,0.999870,NaN,...,3.378792,1052.875,319.500,0.001564,8.679419,2.355004,11.762338,NaN,NaN,NaN
244229,chr22:50799284:G>A,0.477020,0.898633,0.913094,0.961175,0.656112,0.553174,0.686934,0.999363,NaN,...,1.955910,1123.875,130.375,0.001221,1.659057,0.879935,9.713989,NaN,NaN,NaN
244230,chr22:50802392:C>T,0.412290,0.710539,0.991201,0.985088,0.849463,0.843837,0.748275,0.998540,NaN,...,0.540685,651.375,72.000,0.001431,3.483583,0.873050,7.909679,NaN,NaN,NaN


In [12]:
unmatched_keys = set(vcf['variant_name']) - set(df['variant_name'])
unmatched_vcf = vcf[vcf['variant_name'].isin(unmatched_keys)]
unmatched_vcf.to_csv('1000G_SNPs_UNPROCESSED.csv')
unmatched_vcf

,variant_id,chrom,POS,REF,ALT,MAF,CHROM,variant_name
3472,rs61774914,1,-1,C,T,0.03374,chr1,chr1:-1:C>T
3476,rs28525262,1,-1,G,A,0.48360,chr1,chr1:-1:G>A
3477,rs4648756,1,-1,C,G,0.03374,chr1,chr1:-1:C>G
3479,rs4648607,1,-1,C,T,0.03783,chr1,chr1:-1:C>T
3480,rs4648757,1,-1,C,G,0.02658,chr1,chr1:-1:C>G
...,...,...,...,...,...,...,...,...
8563106,rs6009650,22,-1,C,T,0.40080,chr22,chr22:-1:C>T
8563107,rs5769607,22,-1,T,C,0.19020,chr22,chr22:-1:T>C
8563113,rs75012145,22,-1,A,G,0.04397,chr22,chr22:-1:A>G
8563129,rs6009652,22,-1,G,C,0.06033,chr22,chr22:-1:G>C


In [10]:
vcf = pd.read_csv(VCF_PATH, sep='\t')
vcf['CHROM'] = 'chr' + vcf['chrom'].astype(str)
switch_colnames = dict(zip(['pos', 'ref', 'alt', 'rsid'], ['POS', 'REF', 'ALT', 'variant_id']))
vcf.rename(columns=switch_colnames, inplace=True)
vcf['variant_name'] = vcf['CHROM'].astype(str) + ':' + vcf['POS'].astype(str) + ':' + vcf['REF'] + '>' + vcf['ALT']

df_c = df.merge(vcf, on='variant_name', how='left')
df_c

,variant_name,ATAC_quantile,CAGE_quantile,CHIP_HISTONE_quantile,CHIP_TF_quantile,CONTACT_MAPS_quantile,DNASE_quantile,PROCAP_quantile,RNA_SEQ_quantile,SPLICE_JUNCTIONS_quantile,...,SPLICE_JUNCTIONS_raw,SPLICE_SITES_raw,SPLICE_SITE_USAGE_raw,variant_id,chrom,POS,REF,ALT,MAF,CHROM
0,chr1:11008:C>G,0.952545,0.933411,0.984525,0.995798,0.954588,0.986803,0.933411,0.996574,NaN,...,NaN,NaN,NaN,rs575272151,1,11008,C,G,0.08896,chr1
1,chr1:11012:C>G,0.958450,0.963724,0.999058,0.999681,0.974553,0.988931,0.913094,0.997151,NaN,...,NaN,NaN,NaN,rs544419019,1,11012,C,G,0.08896,chr1
2,chr1:13110:G>A,0.628119,0.786165,0.964537,0.971090,0.810542,0.822706,0.883945,0.999703,NaN,...,NaN,NaN,NaN,rs540538026,1,13110,G,A,0.05419,chr1
3,chr1:13116:T>G,0.568981,0.768479,0.982497,0.940451,0.958425,0.692978,0.883945,0.999198,NaN,...,NaN,NaN,NaN,rs62635286,1,13116,T,G,0.18810,chr1
4,chr1:13118:A>G,0.876682,0.784491,0.947050,0.925572,0.938341,0.856617,0.900830,0.999580,NaN,...,NaN,NaN,NaN,rs200579949,1,13118,A,G,0.18810,chr1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8561111,chr22:50798635:T>C,0.868425,0.995975,0.991597,0.990788,0.965955,0.934881,0.963724,0.999887,NaN,...,NaN,NaN,NaN,rs3896457,22,50798635,T,C,0.27100,chr22
8561112,chr22:50798936:A>G,0.884452,0.980378,0.991788,0.995687,0.940957,0.984389,0.879325,0.999870,NaN,...,NaN,NaN,NaN,rs200607599,22,50798936,A,G,0.01227,chr22
8561113,chr22:50799284:G>A,0.477020,0.898633,0.913094,0.961175,0.656112,0.553174,0.686934,0.999363,NaN,...,NaN,NaN,NaN,rs370652263,22,50799284,G,A,0.06953,chr22
8561114,chr22:50802392:C>T,0.412290,0.710539,0.991201,0.985088,0.849463,0.843837,0.748275,0.998540,NaN,...,NaN,NaN,NaN,rs202228854,22,50802392,C,T,0.01738,chr22


In [13]:
df_c.to_csv('1000G_SNPs_PROCESSED_RESULTS.csv')

In [4]:
import pandas as pd
VCF_PATH = '1000G_SNPs_hg38.tsv'  # Path to your VCF file
vcf = pd.read_csv(VCF_PATH, sep='\t')

vcf = vcf[vcf['pos']>=0]
vcf

,rsid,chrom,pos,ref,alt,MAF
0,rs575272151,1,11008,C,G,0.08896
1,rs544419019,1,11012,C,G,0.08896
2,rs540538026,1,13110,G,A,0.05419
3,rs62635286,1,13116,T,G,0.18810
4,rs200579949,1,13118,A,G,0.18810
...,...,...,...,...,...,...
8569057,rs3896457,22,50798635,T,C,0.27100
8569058,rs200607599,22,50798936,A,G,0.01227
8569059,rs370652263,22,50799284,G,A,0.06953
8569060,rs202228854,22,50802392,C,T,0.01738
